In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

Section 1: Loading data and validation

In [8]:
path = '/Users/kaungkhantkyaw/Documents/GitHub/Analyse-US-Domestic-flights/Data/Cleaned/DB1B_Market_model_ready.parquet'
df_model = pd.read_parquet(path)

print(df_model.shape)
display(df_model.head())
display(df_model.sample(5, random_state = 42))
display(df_model.info())

(7189747, 15)


,Origin,Dest,OriginState,DestState,TkCarrier,Passengers,MktDistance,MktCoupons,MktDistanceGroup,MktGeoType,extra_miles,carrier_mismatch,log_passengers,distance_per_coupon,MktFare
0,SAN,ATL,CA,GA,DL,1.0,1892.0,1,4,2,0.0,0,0.693147,1892.0,351.5
1,ATL,SAN,GA,CA,DL,2.0,1892.0,1,4,2,0.0,0,1.098612,1892.0,352.0
2,SAN,ATL,CA,GA,DL,2.0,1892.0,1,4,2,0.0,0,1.098612,1892.0,352.0
3,ATL,SAN,GA,CA,DL,1.0,1892.0,1,4,2,0.0,0,0.693147,1892.0,352.5
4,SAN,ATL,CA,GA,DL,1.0,1892.0,1,4,2,0.0,0,0.693147,1892.0,352.5


,Origin,Dest,OriginState,DestState,TkCarrier,Passengers,MktDistance,MktCoupons,MktDistanceGroup,MktGeoType,extra_miles,carrier_mismatch,log_passengers,distance_per_coupon,MktFare
1799499,GRR,AUS,MI,TX,DL,1.0,1269.0,2,3,2,166.0,1,0.693147,634.5,5.13
7057535,HSV,RDU,AL,NC,DL,1.0,507.0,2,2,2,47.0,0,0.693147,253.5,307.00
4843690,ORD,DEN,IL,CO,WN,1.0,888.0,1,2,2,0.0,0,0.693147,888.0,210.00
6002858,MCO,GJT,FL,CO,UA,1.0,1758.0,2,4,2,36.0,0,0.693147,879.0,522.00
878153,BIL,BFL,MT,CA,AA,1.0,1298.0,2,3,2,395.0,1,0.693147,649.0,194.00


<class 'pandas.DataFrame'>
RangeIndex: 7189747 entries, 0 to 7189746
Data columns (total 15 columns):
 #   Column               Dtype  
---  ------               -----  
 0   Origin               str    
 1   Dest                 str    
 2   OriginState          str    
 3   DestState            str    
 4   TkCarrier            str    
 5   Passengers           float64
 6   MktDistance          float64
 7   MktCoupons           int64  
 8   MktDistanceGroup     int64  
 9   MktGeoType           int64  
 10  extra_miles          float64
 11  carrier_mismatch     int64  
 12  log_passengers       float64
 13  distance_per_coupon  float64
 14  MktFare              float64
dtypes: float64(6), int64(4), str(5)
memory usage: 905.1 MB


None

Section 2: Sample & Separate

Theres almost 7.2 million rows, thus sampling is needed to prevent long training times and using alot of memory.
The sample dataset is also compared to the full set to ensure consistentcy in values.


In [11]:
sample_size = 100_000
target = 'MktFare'

if len(df_model) > sample_size:
    df_sample = df_model.sample(sample_size, random_state = 42).copy()
else:
    df_sample = df_model.copy()

In [12]:
sample_comparison = pd.DataFrame({
    'full_dataset': df_model['MktFare'].describe(),
    'sampled_dataset': df_sample['MktFare'].describe()
})

display(sample_comparison)

quantiles = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]

sample_comparison_quantiles = pd.DataFrame({
    'full_dataset': df_model['MktFare'].quantile(quantiles),
    'sampled_dataset': df_sample['MktFare'].quantile(quantiles)
})

display(sample_comparison_quantiles)


,full_dataset,sampled_dataset
count,7.189747e+06,100000.000000
mean,2.641319e+02,264.463935
std,1.741388e+02,174.320416
min,5.000000e+00,5.000000
25%,1.500000e+02,150.710000
50%,2.331700e+02,233.285000
75%,3.435000e+02,343.500000
max,1.142000e+03,1142.000000


,full_dataset,sampled_dataset
0.01,5.000,5.000
0.05,25.313,25.000
0.25,150.000,150.710
0.50,233.170,233.285
0.75,343.500,343.500
0.95,591.500,593.000
0.99,883.500,884.000


In [14]:
x = df_sample.drop(columns = [target]).copy()
y = df_sample[target].copy()

print("Is target included in x: ", target in x.columns)
print("Features shape: ", x.shape)
print("Target shape: ", y.shape)
print("Target median: ", y.median())

Is target included in x:  False
Features shape:  (100000, 14)
Target shape:  (100000,)
Target median:  233.285


Section 3: Train/Test split

In [15]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)
#x_train is used to train the model, y_train provides the fare values corresponding to it. x_test is the
#unseen values used to test the model, y_test provides the actual fares used to check test predictions.
#80% training data, 20% test data split

print('x_train shape: ', x_train.shape)
print('x_test shape: ', x_test.shape)
print('y_train shape: ', y_train.shape)
print('y_test shape: ', y_test.shape)


x_train shape:  (80000, 14)
x_test shape:  (20000, 14)
y_train shape:  (80000,)
y_test shape:  (20000,)


In [ ]:
cat_cols = ['Origin', 'Dest', 'OriginState', 'DestState', 'TkCarrier', 
'MktDistanceGroup', 'MktGeoType', 'carrier_mismatch']

num_cols = ['Passengers', 'MktDistance', 'MktCoupons', 'extra_miles', 'log_passengers', 'distance_per_coupon']

selected_cols = cat_cols + num_cols

print('Number of selected features: ', len(selected_cols))
print('Number of x columns: ', x.shape[1])

Number of selected features:  14
Number of x columns:  14


In [ ]:
cat_transformer = OneHotEncoder(handle_unknown = 'ignore') #handle_unknown = 'ignore' is used to ignore situations where a rare unseen category appears in test, letting the code run
num_transformer = StandardScaler()

preprocessor = ColumnTransformer(transformers = 
[('categorical', cat_transformer, cat_cols), ('numeric', num_transformer, num_cols)], remainder = 'drop')